In [13]:
import json, math, pickle, re
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from typing import Optional, Dict, List
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import joblib
import warnings
warnings.filterwarnings("ignore")

BASE_DIR = Path("E:/Code/Poliforge/Polyforge-AI/Polyforge-AI")
BRIDGE_DIR = BASE_DIR / "data" / "configs" / "bridges"
BRIDGE_DIR.mkdir(parents=True, exist_ok=True)
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
MODELS_DIR   = BASE_DIR / "models" / "nlp_model_bert"

# 1. Загрузка порогов
with open(MODELS_DIR / "artifacts.pkl", "rb") as f:
    THRESHOLDS = pickle.load(f)["thresholds"]

# 2. Реальный датасет (для сверки)
df_real = pd.read_csv(DATA_RAW_DIR / "polymers_with_names_predicted_selfies.csv")
df_real["Polymer_SMILES"] = df_real["Polymer_SMILES"].str.strip()

# 3. Модель LightGBM для предсказания свойств
FINGERPRINT_MODEL_PATH = BASE_DIR / "models" / "fingerprint_property_model.pkl"
artifacts = joblib.load(FINGERPRINT_MODEL_PATH)
lgb_models = artifacts['models']
fp_scaler = artifacts['scaler']
fp_property_cols = artifacts['property_cols']
feature_names = artifacts.get('feature_names', [f"fp_{i}" for i in range(1024)])

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

In [24]:
def predict_properties_from_smiles(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = np.array(gen.GetFingerprint(mol)).reshape(1, -1)
    fp_df = pd.DataFrame(fp, columns=feature_names)
    preds_scaled = [lgb_models[prop].predict(fp_df)[0] for prop in fp_property_cols]
    preds_scaled = np.array(preds_scaled).reshape(1, -1)
    preds = fp_scaler.inverse_transform(preds_scaled).flatten()
    return dict(zip(fp_property_cols, preds))

# 4. Загрузка требований из моста NLP→Gen
def load_nlp_requirements(path: Path) -> dict:
    with open(path, encoding='utf-8') as f:
        return json.load(f)

bridge = load_nlp_requirements(BRIDGE_DIR / "nlp_to_gen_bridge.json")

if "user_specified" in bridge and "auto_filled" in bridge:
    user_specified = bridge["user_specified"]
    auto_filled = bridge["auto_filled"]
elif "requirements" in bridge:
    reqs = bridge["requirements"]
    if isinstance(reqs, list) and reqs and isinstance(reqs[0], dict):
        user_specified = {r["param"]: r for r in reqs if r.get("user_specified", False)}
        auto_filled = {r["param"]: r for r in reqs if not r.get("user_specified", False)}
    else:
        raise ValueError("Неизвестный формат requirements")
elif all(isinstance(v, dict) for v in bridge.values()):
    user_specified = bridge
    auto_filled = {}
else:
    raise KeyError(f"Неизвестная структура моста: {list(bridge.keys())}")

def prepare_requirement_strict(param, req, thresholds, is_user):
    t = thresholds.get(param)
    if t is None:
        L, U = req.get("min", req["target_value"]), req.get("max", req["target_value"])
        sigma = (U - L) * 0.2 if U > L else 1.0
    else:
        full_range = t["max"] - t["min"]
        target = req["target_value"]
        if is_user:
            L = target - 0.02 * full_range
            U = target + 0.02 * full_range
            sigma = 0.02 * full_range
        else:
            L = target - 0.10 * full_range
            U = target + 0.10 * full_range
            sigma = 0.05 * full_range
        L, U = max(L, t["min"]), min(U, t["max"])
    return {
        "param": param, "target": req["target_value"],
        "L": L, "U": U, "sigma": sigma,
        "is_user": is_user, "category": req.get("category", "medium")
    }

user_reqs = [prepare_requirement_strict(p, r, THRESHOLDS, True) for p, r in user_specified.items() if prepare_requirement_strict(p, r, THRESHOLDS, True)]
auto_reqs = [prepare_requirement_strict(p, r, THRESHOLDS, False) for p, r in auto_filled.items() if prepare_requirement_strict(p, r, THRESHOLDS, False)]
all_reqs = user_reqs + auto_reqs
# Определяем структуру
if isinstance(bridge, dict):
    # Если есть ключ 'meta', это старый формат
    if 'meta' in bridge:
        meta = bridge['meta']
        if 'user_specified' in bridge and 'auto_filled' in bridge:
            user_specified = bridge['user_specified']
            auto_filled = bridge['auto_filled']
        elif 'requirements' in bridge:
            reqs = bridge['requirements']
            user_specified = {r["param"]: r for r in reqs if r.get("user_specified", False)}
            auto_filled = {r["param"]: r for r in reqs if not r.get("user_specified", False)}
        else:
            raise ValueError("Неизвестная структура моста с ключом 'meta'")
    else:
        # Нет 'meta' – считаем весь словарь списком пользовательских требований
        # (можно также проверить, есть ли внутри ключ 'target_value')
        user_specified = {}
        auto_filled = {}
        for key, val in bridge.items():
            if isinstance(val, dict) and 'target_value' in val:
                user_specified[key] = val
        meta = {"original_query": "запрос из моста (без meta)"}
else:
    raise TypeError("Мост должен быть словарем")

print(f"Пользовательские требования: {len(user_specified)}, авто: {len(auto_filled)}")
print(f"Загружено требований: {len(user_reqs)} пользовательских, {len(auto_reqs)} авто")

Пользовательские требования: 37, авто: 0
Загружено требований: 37 пользовательских, 0 авто


In [25]:
def load_polymers(path, requirements):
    df = pd.read_csv(path)
    # ищем столбец со SMILES
    smile_col = None
    for col in ["smiles", "Polymer_SMILES", "SELFIES"]:
        if col in df.columns:
            smile_col = col
            break
    if smile_col is None:
        raise KeyError("Не найдена колонка SMILES в gen_to_val_bridge.csv")
    df = df.rename(columns={smile_col: "smiles"})
    # оставляем только нужные параметры
    param_cols = []
    for req in requirements:
        param = req["param"]
        if param in df.columns:
            param_cols.append(param)
        elif f"{param}_target" in df.columns:
            df = df.rename(columns={f"{param}_target": param})
            param_cols.append(param)
    cols = ["smiles"] + param_cols
    return df[cols]

polymers_df = load_polymers(BRIDGE_DIR / "gen_to_val_bridge.csv", all_reqs)
print(f"Загружено полимеров: {len(polymers_df)}")


Загружено полимеров: 5


In [26]:
def check_smiles_valid(smiles):
    return Chem.MolFromSmiles(smiles) is not None

def find_real_properties(smiles, df_real, required_params):
    match = df_real[df_real["Polymer_SMILES"] == smiles]
    if match.empty:
        return None
    row = match.iloc[0]
    props = {}
    for param in required_params:
        if param in row and pd.notna(row[param]):
            props[param] = float(row[param])
    return props

def evaluate_parameter(polymer_value, req):
    if polymer_value is None or math.isnan(polymer_value):
        return 0.1 if req["is_user"] else 0.5
    L, U, sigma = req["L"], req["U"], req["sigma"]
    if L <= polymer_value <= U:
        return 1.0
    dist = L - polymer_value if polymer_value < L else polymer_value - U
    return math.exp(-0.5 * (dist / sigma) ** 2)

def compute_polymer_score(row, user_reqs, auto_reqs, df_real, sigma_rel=0.2):
    smiles = row.get("smiles", "")
    if not check_smiles_valid(smiles):
        return {"overall_score": 0.0, "details": {}, "notes": "Invalid SMILES"}

    # 1. Соответствие требованиям NLP
    scores = {}
    for req in user_reqs + auto_reqs:
        scores[req["param"]] = evaluate_parameter(row.get(req["param"]), req)

    # среднее геометрическое с весами
    log_sum = 0.0
    total_weight = 0.0
    for req in user_reqs:
        s = max(scores[req["param"]], 1e-6)
        log_sum += 1.0 * math.log(s)
        total_weight += 1.0
    for req in auto_reqs:
        s = max(scores[req["param"]], 1e-6)
        log_sum += 0.3 * math.log(s)
        total_weight += 0.3
    base_score = math.exp(log_sum / total_weight) if total_weight > 0 else 0.0

    # 2. Штраф от БД
    db_penalty = 1.0
    required_params = [r["param"] for r in user_reqs + auto_reqs]
    real_props = find_real_properties(smiles, df_real, required_params)
    if real_props:
        deviations = 0
        for param in required_params:
            if param in real_props and param in row:
                stated = row[param]
                if stated is None or math.isnan(stated):
                    continue
                actual = real_props[param]
                rel_diff = abs(stated - actual) / abs(actual) if actual != 0 else abs(stated - actual)
                if rel_diff > 0.20:
                    deviations += 1
        db_penalty = max(0.5, 1.0 - 0.05 * deviations)

    # 3. Реализм через LightGBM
    realism_score = 1.0
    pred_props = predict_properties_from_smiles(smiles)
    if pred_props:
        factors = []
        for req in user_reqs + auto_reqs:
            param = req["param"]
            if param not in pred_props or param not in row:
                continue
            stated = row[param]
            predicted = pred_props[param]
            if stated is None or math.isnan(stated) or predicted == 0:
                continue
            rel_err = abs(stated - predicted) / abs(predicted)
            factor = math.exp(-0.5 * (rel_err / sigma_rel) ** 2)
            factors.append(max(factor, 1e-12))
        if factors:
            log_prod = sum(math.log(f) for f in factors)
            realism_score = math.exp(log_prod / len(factors))

    overall = base_score * db_penalty * realism_score
    notes = []
    if db_penalty < 1.0: notes.append(f"DB mismatch penalty: {db_penalty:.2f}")
    if realism_score < 0.999: notes.append(f"Realism: {realism_score:.4f}")
    return {
        "overall_score": overall, "details": scores,
        "db_penalty": db_penalty, "realism_score": realism_score,
        "notes": "; ".join(notes) if notes else "OK"
    }

In [27]:
def evaluate_all_polymers(df, user_reqs, auto_reqs, df_real):
    results = []
    for _, row in df.iterrows():
        res = compute_polymer_score(row, user_reqs, auto_reqs, df_real)
        results.append({**row.to_dict(), "score": res["overall_score"],
                        "realism_score": res["realism_score"], "notes": res["notes"]})
    return pd.DataFrame(results).sort_values("score", ascending=False)

# 7. Запуск оценки
evaluated_df = evaluate_all_polymers(polymers_df, user_reqs, auto_reqs, df_real)
print("Топ-5 полимеров:")
print(evaluated_df[["smiles", "score", "realism_score", "notes"]].head(5))

Топ-5 полимеров:
                                              smiles     score  realism_score  \
4  [*]c1ccc(SC(=O)c2ccc(S(=O)(=O)c3ccc(-c4ccc5c(c...  0.177983       0.698100   
3  [*]Nc1cc(-c2cc(C3CCC(N4C(=O)c5ccc(N6C(=O)c7ccc...  0.137283       0.538465   
0  [*]Oc1cc([*])cc(-c2ccc(S(=O)(=O)c3ccc(N4C(=O)c...  0.005620       0.022042   
1  [*]Oc1ccc2c(c1)C(=O)N(c1cccc(-c3ccc(S(=O)(=O)c...  0.004498       0.017644   
2  [*]c1nc(-c2ccc(P(=O)(c3ccccc3)c3ccc(-c4ccc5c(c...  0.002244       0.008802   

             notes  
4  Realism: 0.6981  
3  Realism: 0.5385  
0  Realism: 0.0220  
1  Realism: 0.0176  
2  Realism: 0.0088  


In [29]:
# Сохранение отчётов
evaluated_df.to_csv(output_dir / "validation_full_results.csv", index=False)
top10 = evaluated_df.head(10)
top10.to_csv(output_dir / "validation_top.csv", index=False)

with open(output_dir / "summary.txt", "w", encoding="utf-8") as f:
    f.write(f"Оценено {len(evaluated_df)} полимеров\n")
    # Безопасно извлекаем original_query
    original_query = meta.get("original_query", "не указан")
    f.write(f"Запрос: {original_query}\n\n")
    for i, (_, row) in enumerate(top10.iterrows()):
        f.write(f"{i+1}. {row['smiles']} – Score: {row['score']:.4f} ({row.get('notes', '')})\n")

print("Отчёты сохранены в", output_dir)

Отчёты сохранены в E:\Code\Poliforge\Polyforge-AI\Polyforge-AI\data\configs\bridges


In [22]:
test_smiles = "CC(C)(c1ccc(*)cc1)c1ccc(OCC(=O)OC(=O)c2cccc(c2)C(=O)OC(=O)CO*)cc1"
props = predict_properties_from_smiles(test_smiles)
print("Предсказанные свойства:", props)

Предсказанные свойства: {'Egc': np.float64(4.124098217782926), 'Egb': np.float64(3.7913736334092376), 'Eib': np.float64(3.5936963706010445), 'CED': np.float64(97.08735344827699), 'Ei': np.float64(5.926650249823525), 'Eea': np.float64(1.409086227445254), 'nc': np.float64(1.8411948746602058), 'ne': np.float64(1.57766184899455), 'Xc': np.float64(30.56333559172544), 'Xe': np.float64(29.427336995150984), 'epse_6.0': np.float64(3.5043099402684867), 'epsc': np.float64(4.257760426545684), 'epse_3.0': np.float64(3.812825709350227), 'epse_1.78': np.float64(4.013436735606909), 'epse_15.0': np.float64(2.512676095968456), 'epse_4.0': np.float64(3.726696086483013), 'epse_5.0': np.float64(3.652505646542651), 'epse_2.0': np.float64(3.934549498557509), 'epse_9.0': np.float64(3.077196928399939), 'epse_7.0': np.float64(3.366592073018857), 'epsb': np.float64(12.63766513414489), 'TSb': np.float64(86.28380502050848), 'TSy': np.float64(84.24976537176083), 'YM': np.float64(2466.9690772358126), 'permCH4': np.f

In [30]:
# Тестовые SMILES из датасета
test_smiles_list = [
    '*c1ccc(cc1)-c1ccc(cc1)-c1ccc(cc1)-c1ccc(Oc2ccc(cc2)-c2ccc(cc2)-c2ccc(*)cc2)cc1',
    '*c1ccc(s1)-c1ccc(cc1)-c1ccc(Oc2ccc(cc2)C(=O)c2ccc(Oc3ccc(cc3)C(=O)c3ccc(*)cc3)cc2)cc1',
    '*c1ccc(OCCC(=O)c2ccc(cc2)C(=O)c2ccc(cc2)C(=O)c2ccc(cc2)-c2cccc(c2)-c2ccc(*)cc2)cc1',
    '*c1nc2cc3sc(nc3cc2s1)-c1ccc(Oc2ccc(cc2)-c2cccc(c2)C(=O)Nc2ccc(*)cc2)cc1',
    '*c1ccc(Oc2ccc(cc2)-c2ccc(cc2)-c2ccc(cc2)-c2ccc(cc2)C(=O)c2ccc(cc2)C(=O)c2ccc(*)cc2)cc1',
    '*c1nc2ccc(cc2o1)-c1ccc(Oc2ccc(cc2)C(=O)c2ccc(C(=O)c3ccc(Oc4ccc(cc4)C(=O)c4ccccc(*)cc4)cc3)cc2)cc1',
    'COc1cc(ccc1*)-c1ccc(NC(=O)c2ccc(cc2)-c2ccc(cc2)-c2cccc(c2)C(=O)c2ccc(O*)cc2)cc1',
    '*c1ccc(cc1)-c1ccc(cc1)-c1ccc(cc1)-c1ccc2C(=O)N(*)C(=O)c2c1',
    '*c1ccc(cc1)-c1ccc(Oc2ccc(cc2)S(=O)(=O)c2ccc(cc2)-c2ccc(Oc3ccc(cc3)C(=O)c3ccc(*)cc3)cc2)cc1',
    '*c1nc2cc3sc(nc3cc2s1)-c1ccc(Oc2ccc(cc2)C(=O)c2ccc(cc2)C(=O)Nc2ccc(*)cc2)cc1',
    '*Oc1ccc(Oc2ccc(cc2)C(=O)c2ccc(cc2)C(=O)c2ccc(*)cc2)cc1',
]

# Формируем DataFrame, где заявленные свойства = предсказанные
rows = []
for smi in test_smiles_list:
    preds = predict_properties_from_smiles(smi)
    if preds is None:
        continue
    row = {"smiles": smi}
    row.update(preds)          # все свойства как _target
    rows.append(row)

test_df = pd.DataFrame(rows)

# Прогоняем через валидатор
test_evaluated = evaluate_all_polymers(test_df, user_reqs, auto_reqs, df_real)

print("Тест полимеров из датасета (свойства = предсказания LightGBM):")
print(test_evaluated[["smiles", "score", "realism_score", "notes"]].head(5))

Тест полимеров из датасета (свойства = предсказания LightGBM):
                                              smiles     score  realism_score  \
7  *c1ccc(cc1)-c1ccc(cc1)-c1ccc(cc1)-c1ccc2C(=O)N...  0.384279            1.0   
4  *c1ccc(Oc2ccc(cc2)-c2ccc(cc2)-c2ccc(cc2)-c2ccc...  0.313945            1.0   
5  *c1nc2ccc(cc2o1)-c1ccc(Oc2ccc(cc2)C(=O)c2ccc(C...  0.286231            1.0   
3  *c1nc2cc3sc(nc3cc2s1)-c1ccc(Oc2ccc(cc2)-c2cccc...  0.240368            1.0   
2  *c1ccc(OCCC(=O)c2ccc(cc2)C(=O)c2ccc(cc2)C(=O)c...  0.226739            1.0   

  notes  
7    OK  
4    OK  
5    OK  
3    OK  
2    OK  
